In [ ]:
# 02_gis_water_extent.ipynb  (rewritten to avoid "Too many concurrent aggregations")

# ---- Cell 1: Install and initialize ----
!pip install -q earthengine-api geemap geopandas

import ee, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ee.Authenticate()          # skip if already authenticated in this session
PROJECT_ID = 'terraworks-511519'
ee.Initialize(project=PROJECT_ID)
print('Earth Engine ready')

# ---- Cell 2: Settings ----
UJANI_LAT, UJANI_LON = 18.0, 75.1          # approximate; replace with exact boundary when available
centre = ee.Geometry.Point([UJANI_LON, UJANI_LAT])
aoi = centre.buffer(10000).bounds()         # same 10 km study area as notebook 01

START_YEAR, END_YEAR = 2021, 2024          # start with a short range; extend once it works
NDWI_THRESHOLD = 0.2                        # calibrate in the check cell
VV_THRESHOLD_DB = -18                       # open water usually below about -15 to -20 dB
CHECKPOINT_CSV = 'ujani_water_checkpoint.csv'

month_starts = pd.date_range(f'{START_YEAR}-01-01', f'{END_YEAR}-12-01', freq='MS') \
                 .strftime('%Y-%m-%d').tolist()
print(f'{len(month_starts)} months to process')
# ---- Cell 3: Collections and masking ----
s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40)))

s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(aoi)
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV'))

def mask_s2(img):
    """Keep clear pixels using the Scene Classification Layer (SCL)."""
    scl = img.select('SCL')
    clear = (scl.neq(0).And(scl.neq(3)).And(scl.neq(8))
                 .And(scl.neq(9)).And(scl.neq(10)))
    return img.updateMask(clear).copyProperties(img, ['system:time_start'])

# ---- Cell 4: One-month functions (each makes ONE server call) ----
def s2_month(start_str):
    """Returns image count and water area (km²) for one month, Sentinel-2 NDWI."""
    start = ee.Date(start_str)
    col = s2_col.filterDate(start, start.advance(1, 'month'))
    n = col.size()

    # Median of masked collection
    composite = col.map(mask_s2).median()

    # Check if the composite has any bands (in case of total cloud cover)
    band_names = composite.bandNames()
    has_bands = band_names.contains('B3').And(band_names.contains('B8'))

    # Conditional execution inside Earth Engine
    area_m2 = ee.Algorithms.If(
        has_bands,
        ee.Image.pixelArea().updateMask(
            composite.normalizedDifference(['B3', 'B8']).gt(NDWI_THRESHOLD)
        ).reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                       scale=10, maxPixels=1e10,
                       bestEffort=True, tileScale=4).get('area'),
        None
    )

    return ee.Dictionary({'n': n, 'area': area_m2}).getInfo()

def s1_month(start_str):
    """Returns image count and water area (km²) for one month, Sentinel-1 radar."""
    start = ee.Date(start_str)
    col = s1_col.filterDate(start, start.advance(1, 'month'))
    n = col.size()

    # Check if we have Sentinel-1 images
    has_images = n.gt(0)

    area_m2 = ee.Algorithms.If(
        has_images,
        ee.Image.pixelArea().updateMask(
            col.median().lt(VV_THRESHOLD_DB)
        ).reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                       scale=10, maxPixels=1e10,
                       bestEffort=True, tileScale=4).get('area'),
        None
    )
    return ee.Dictionary({'n': n, 'area': area_m2}).getInfo()

def call_with_retry(fn, arg, retries=5):
    """Retries when Earth Engine reports quota or concurrency limits."""
    for attempt in range(retries):
        try:
            return fn(arg)
        except Exception as e:
            msg = str(e).lower()
            if any(k in msg for k in ['concurrent', 'quota', 'rate', 'too many']) and attempt < retries - 1:
                wait = 10 * (2 ** attempt)
                print(f'  limit reached, waiting {wait}s and retrying...')
                time.sleep(wait)
            else:
                raise
# ---- Cell 5: Run month by month with checkpointing ----
import ee, time, os
import numpy as np
import pandas as pd

# Redefining the function locally to bypass non-existent ee.Boolean wrapper classes using numerical logic
def s2_month_fixed(start_str):
    """Returns image count and water area (km²) for one month, Sentinel-2 NDWI."""
    start = ee.Date(start_str)
    col = s2_col.filterDate(start, start.advance(1, 'month'))
    n = col.size()

    # Median of masked collection
    composite = col.map(mask_s2).median()

    # Check if the composite has any bands (in case of total cloud cover)
    band_names = composite.bandNames()

    # Numerical logic: (index of B3 >= 0) * (index of B8 >= 0) is 1 if both exist, else 0
    has_bands = band_names.indexOf('B3').gte(0).multiply(band_names.indexOf('B8').gte(0))

    # Conditional execution inside Earth Engine
    area_m2 = ee.Algorithms.If(
        has_bands,
        ee.Image.pixelArea().updateMask(
            composite.normalizedDifference(['B3', 'B8']).gt(NDWI_THRESHOLD)
        ).reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                       scale=10, maxPixels=1e10,
                       bestEffort=True, tileScale=4).get('area'),
        None
    )

    return ee.Dictionary({'n': n, 'area': area_m2}).getInfo()

# Override the function name
s2_month = s2_month_fixed

# Progress is saved after every month, so if the run stops you can re-run
# this cell and it resumes from where it left off.
if os.path.exists(CHECKPOINT_CSV):
    done = pd.read_csv(CHECKPOINT_CSV)
    results = done.to_dict('records')
    done_months = set(done['month'])
    print(f'Resuming: {len(done_months)} months already saved')
else:
    results, done_months = [], set()

for start in month_starts:
    month = start[:7]
    if month in done_months:
        continue

    s2 = call_with_retry(s2_month, start)
    time.sleep(1)                     # small pause to stay under concurrency limits
    s1 = call_with_retry(s1_month, start)
    time.sleep(1)

    results.append({
        'month': month,
        's2_images': s2['n'],
        's2_water_km2': (s2['area'] or 0) / 1e6 if s2['n'] > 0 else np.nan,
        's1_images': s1['n'],
        's1_water_km2': (s1['area'] or 0) / 1e6 if s1['n'] > 0 else np.nan,
    })
    pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False)
    print(f'{month}: S2 {results[-1]["s2_water_km2"]} km" | S1 {results[-1]["s1_water_km2"]} km"')

df = pd.DataFrame(results).sort_values('month').reset_index(drop=True)
df['date'] = pd.to_datetime(df['month'])
print(f"\nMonths with Sentinel-2 data: {df['s2_water_km2'].notna().sum()} of {len(df)}")
print(f"Months with Sentinel-1 data: {df['s1_water_km2'].notna().sum()} of {len(df)}")

# ---- Cell 6: Plot ----
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['date'], df['s2_water_km2'], 'o-', label='Sentinel-2 (NDWI, optical)', alpha=0.8)
ax.plot(df['date'], df['s1_water_km2'], 's-', label='Sentinel-1 (VV radar)', alpha=0.8)
ax.set_title('Monthly water extent: Ujani area')
ax.set_ylabel('Water area (km²)')
ax.set_xlabel('Month')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ---- Cell 7: Calibrate the NDWI threshold for one clear month (4 calls, sequential) ----
CHECK_MONTH = '2023-10-01'     # post-monsoon, usually clear; change if needed
check_start = ee.Date(CHECK_MONTH)
check_img = (s2_col.filterDate(check_start, check_start.advance(1, 'month'))
             .map(mask_s2).median())
check_ndwi = check_img.normalizedDifference(['B3', 'B8'])

for thr in [0.0, 0.1, 0.2, 0.3]:
    area = call_with_retry(
        lambda t: (ee.Image.pixelArea().updateMask(check_ndwi.gt(t))
                   .reduceRegion(ee.Reducer.sum(), aoi, 10, maxPixels=1e10,
                                 bestEffort=True, tileScale=4)
                   .get('area')).getInfo(),
        thr)
    print(f'NDWI > {thr}: {(area or 0) / 1e6:.2f} km²')
    time.sleep(1)

# ---- Cell 8: Visual check ----
import geemap
Map = geemap.Map(center=[UJANI_LAT, UJANI_LON], zoom=12)
Map.addLayer(check_img, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'True colour')
Map.addLayer(check_ndwi.gt(NDWI_THRESHOLD).selfMask(),
             {'palette': ['blue']}, f'Water mask (NDWI > {NDWI_THRESHOLD})')
Map

# ---- Cell 9: Export ----
OUTPUT_CSV = 'ujani_water_extent_monthly.csv'
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {OUTPUT_CSV}')